In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('sales')
display(df)

In [0]:
import logging

logger = get_logger("silver_sales")

try:
    logger.info("Starting Silver Sales transformation")


    # ============================================================
    # 1. Data type conversion and string trimming
    # ============================================================

    logger.info("Updating data types of the columns")
    logger.info("Trimming leading/trailing spaces from string columns")

    df = df.select(
        trim(col("order_id").try_cast("string")).alias("order_id"),
        trim(col("customer_id").try_cast("string")).alias("customer_id"),
        trim(col("product_id").try_cast("string")).alias("product_id"),
        trim(col("store_id").try_cast("string")).alias("store_id"),
        col("order_date").try_cast("date").alias("order_date"),
        col("quantity").try_cast("integer").alias("quantity"),
        col("unit_price").try_cast("decimal(10,2)").alias("unit_price"),
        col("discount").try_cast("decimal(10,2)").alias("discount"),
        trim(col("payment_method").try_cast("string")).alias("payment_method"),
        col("_ingestion_timestamp").try_cast("timestamp").alias("_ingestion_timestamp"),
        col("_source_file").try_cast("string").alias("_source_file")
    )

    logger.info("Data type conversion and string trimming completed")


    # ============================================================
    # 2. Remove duplicate orders
    # ============================================================

    before_count = df.count()

    duplicate_count = (
        before_count -
        df.dropDuplicates(["order_id"]).count()
    )

    logger.info(f"Duplicate orders found: {duplicate_count}")

    df = df.dropDuplicates(["order_id"])

    logger.info("Duplicate orders removed")


    # ============================================================
    # 3. Remove invalid quantity/unit_price
    # ============================================================

    logger.info("Removing records where quantity or unit_price is <= 0")

    before_count = df.count()

    df = df.where(
        (col("quantity") > 0) &
        (col("unit_price") > 0)
    )

    after_count = df.count()

    records_dropped = before_count - after_count

    logger.info(f"Records dropped due to invalid quantity/unit_price: {records_dropped}")


    # ============================================================
    # 4. Remove records with null order_date
    # ============================================================

    logger.info("Removing records with null order_date")

    before_count = df.count()

    df = df.where(
        ~isnull(col("order_date"))
    )

    after_count = df.count()

    records_dropped = before_count - after_count

    logger.info(f"Records dropped due to null order_date: {records_dropped}")


    # ============================================================
    # 5. Handle null discounts
    # ============================================================

    null_discount_count = df.where(
        isnull(col("discount"))
    ).count()

    logger.info(
        f"Null discount records found: {null_discount_count}"
    )

    if null_discount_count > 0:

        logger.info("Filling null discounts with 0")

        df = df.fillna(
            0,
            ["discount"]
        )

        logger.info("Null discounts filled with 0")

    else:

        logger.info("No null values found in discount column")


    # ============================================================
    # 6. Handle null payment methods
    # ============================================================

    null_payment_count = df.where(
        isnull(col("payment_method"))
    ).count()

    logger.info(
        f"Null payment_method records found: {null_payment_count}"
    )

    if null_payment_count > 0:

        logger.info(
            'Filling null payment_method values with "UNKNOWN"'
        )

        df = df.fillna(
            "UNKNOWN",
            ["payment_method"]
        )

        logger.info(
            'Null payment_method values filled with "UNKNOWN"'
        )

    else:

        logger.info(
            "No null values found in payment_method column"
        )
        
    # ============================================================
    # 7. Replacing PROD prefix from product_id with "P" & STORE prefix from store_id with "S" & CUST prefix from customer_id with "C"
    # ============================================================
    
    df= df.withColumn('product_id', regexp_replace(col('product_id'),"^PROD" ,"P"))\
        .withColumn('store_id', regexp_replace(col('store_id'),"^STORE" ,"S"))\
        .withColumn('customer_id', regexp_replace(col('customer_id'),"^CUST" ,"C"))

    df.printSchema()

    display(df)
        
    # ============================================================
    # 8. Create schema
    # ============================================================

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )


    # ============================================================
    # 9. Save Silver table
    # ============================================================

    logger.info("Saving Silver Sales table")

    save_table(
        df,
        "sales_clean"
    )

    logger.info("Silver Sales table saved successfully")

    logger.info("Silver Sales transformation completed successfully")

except Exception:
    logger.exception("Silver Sales transformation failed")